# Projet Recherche Opérationnelle

**Classe :** FISA A3 - Informatique  
**Groupe :** NOIRET Robin, LALOUANI Jalil, KLEBER Mattéo, CAPARROS Martin, UPJOHN Max  
**Période :** 03/2026 - 04/2026

---


# 1. Identification du problème

Le problème posé par l'ADEME consiste à l'**optimisation de tournées logistiques** : un véhicule part d'un dépôt, dessert un ensemble de points d'arrêt, puis retourne à son point d'origine.

L'objectif est de **minimiser la distance totale du trajet**, tout en respectant un ensemble de **contraintes**.

Ce problème est une **variante du problème du voyageur de commerce** (*Travelling Salesman Problem*, TSP)[$^5$](#52-références-scientifiques) [$^6$](#52-références-scientifiques). La différence fondamentale avec le TSP classique réside dans l'ajout de contraintes supplémentaires :
- certaines routes sont **interdites ou coûteuses** (travaux, restrictions de gabarit, zones à faibles émissions) ;
- certaines visites ont des **dépendances d'ordre** (une livraison doit précéder la collecte associée).

# 2. Définition mathématique du problème

## 2.1 Données du problème

On modélise le réseau routier par un graphe orienté pondéré :

$$G = (V, E, c)$$

avec :

- $V = \{v_0, v_1, \ldots, v_n\}$ : l'ensemble des **nœuds** (villes), où $v_0$ désigne le **dépôt** (point de départ et d'arrivée)
- $E = \{(i,j) \mid i \in V,\ j \in V,\ i \neq j\}$ : l'ensemble des **arêtes** (routes)
- $c'_{ij} = \alpha_{ij} \cdot c_{ij}$ : la **fonction de coût**, associant à chaque arête une distance. On ajoute une notion de coefficient (BigM)
> $\quad \quad \quad \alpha_{ij} = \begin{cases} 1 & \text{route classique} \\ [2,\ 10] & \text{route dégradée} \\ [1000,\ 10000] & \text{route interdite} \end{cases}$

On définit également les ensembles suivants pour modéliser nos contraintes supplémentaires :

- $F \subseteq E$ : l'ensemble des **arêtes interdites** (routes bloquées, trop coûteuses ou inaccessibles)
- $D \subseteq V^2$ : l'ensemble des **paires de dépendance** de la forme $(i, j)$, signifiant que la ville $i$ doit être visitée avant la ville $j$ (livraison avant collecte)

> **Note :** Le graphe ne contient pas nécessairement de cycle hamiltonien, il faudra mettre en place une vérification/condition à la génération/regénération

## 2.2 Variables de décision

**Variables binaires de sélection des arêtes :**

$$x_{ij} \in \{0, 1\} \quad \forall (i,j) \in E$$

$x_{ij} = 1$ si et seulement si l'arête $(i,j)$ est empruntée dans la tournée, $0$ sinon.

**Variables de position (rang dans la tournée) :**

$$p_i \in \{1, \ldots, n\} \quad \forall i \in V \setminus \{v_0\}$$

$p_i$ représente le rang (ordre de passage) de la ville $i$ dans la tournée.

## 2.3 Fonction objectif

On cherche à **minimiser la distance totale** de la tournée :

$$\min \sum_{i \in V} \sum_{\substack{j \in V \\ j \neq i}} c'_{ij} \cdot x_{ij}$$

## 2.4 Contraintes

#### (C1) Chaque ville est visitée exactement une fois

$$\sum_{j \in V,\ j \neq i} x_{ij} = 1 \quad \forall i \in V  \quad \quad \quad

\sum_{i \in V,\ i \neq j} x_{ij} = 1 \quad \forall j \in V$$

#### (C2) Coût ou restriction de passage sur certaines arêtes

Certaines routes peuvent être dégradées (travaux, détours) ou totalement inaccessibles. Plutôt qu'une interdiction stricte, on modélise ces restrictions par un **coefficient de pénalité** $\alpha_{ij}$ appliqué au coût de chaque arête :

$$c'_{ij} = \alpha_{ij} \cdot c_{ij}$$

$$\alpha_{ij} = \begin{cases} 1 & \text{route classique} \\ [2,\ 10] & \text{route dégradée} \\ [1000,\ 10000] & \text{route interdite} \end{cases}$$

> **Note :** Les arêtes "interdites" ne sont pas exclues par une contrainte dure, mais leur coût ajusté garantit qu'elles ne seront jamais empruntées par une solution optimale, étant donné l'échelle classique des coûts ($c_{ij} \in [1, 10]$).

#### (C3) Contraintes de dépendances de visite

Pour chaque paire de dépendance $(i, j) \in D$, la ville $i$ doit être visitée **avant** la ville $j$ :

$$p_i + 1 \leq p_j \quad \forall (i,j) \in D$$

> **Note :** Cette formule garantit un écart d'au moins une position entre les deux villes dans l'ordre de visite.

#### (C4) Élimination des sous-tours — Formulation MTZ

Pour éliminer les sous-tours (circuits ne passant pas par le dépôt), on utilise la formulation de Miller-Tucker-Zemlin (MTZ)[$^7$](#52-références-scientifiques) [$^8$](#52-références-scientifiques)  :

$$p_i - p_j + n \cdot x_{ij} \leq n - 1 \quad \forall (i,j) \in E,\ i \neq v_0,\ j \neq v_0$$

**Explication :**
- Si l'arête $(i,j)$ est utilisée ($x_{ij} = 1$) : la contrainte devient $p_i - p_j \leq -1$, soit $p_j \geq p_i + 1$. La ville $j$ est donc visitée **immédiatement après** la ville $i$ dans la tournée, ce sont donc deux villes consécutives. (cette contrainte représente l'élimination des sous-tours et non la dépendance de visite)
- Si l'arête $(i,j)$ n'est pas utilisée ($x_{ij} = 0$) : la contrainte devient $p_i - p_j \leq n - 1$, ce qui est toujours satisfait puisque les rangs appartiennent à $\{1, \dots, n\}$. La contrainte est relâchée.

## 2.5 Récapitulation (programmation linéaire)

On formule le problème comme un **programme linéaire en nombres entiers** (*Integer Linear Programming*, ILP). Les variables $x_{ij}$ sont binaires et les variables $p_i$ sont entières. Toutes les contraintes sont linéaires en ces variables.

$$
\text{PL} : \begin{cases}
\min & \displaystyle\sum_{i \in V} \sum_{\substack{j \in V \\ j \neq i}} c'_{ij} \cdot x_{ij} \\[10pt]
\ & \displaystyle\sum_{j \in V,\, j \neq i} x_{ij} = 1 \quad \forall i \in V \quad \text{(C1a)} \\[10pt]
& \displaystyle\sum_{i \in V,\, i \neq j} x_{ij} = 1 \quad \forall j \in V \quad \text{(C1b)} \\[10pt]
& p_i + 1 \leq p_j \quad \forall (i,j) \in D \quad \text{(C3)} \\[10pt]
& p_i - p_j + n \cdot x_{ij} \leq n - 1 \quad \forall (i,j) \in E,\ i \neq v_0,\ j \neq v_0 \quad \text{(C4)} \\[10pt]
& x_{ij} \in \{0, 1\} \\[10pt]
& p_i \in \{1, \ldots, n\} \\[10pt]
\end{cases}
$$

# 3. Analyse de la complexité

> **Note :** A développer

Notre variante du TSP (incluant des précédences et des arêtes interdites) est une extension du problème classique du voyageur de commerce, dont la Incomplétude a été prouvée par Karp en 1972 dans son ouvrage [$^{10}$](#52-références-scientifiques).

Pour démontrer que notre problème est NP-difficile, il suffit d'appliquer le principe de restriction.</br> Si l'on considère une instance de notre problème sans **aucune contrainte de précédence ni arête interdite**, on retrouve exactement le TSP standard. Notre problème contenant le TSP comme cas particulier, il est donc au moins aussi complexe à résoudre.

Cette nature NP-difficile justifie l'utilisation de méthodes approchées (heuristiques et métaheuristiques) pour les grands réseaux routiers, là où les méthodes exactes deviendraient inopérantes face à l'explosion combinatoire.

# 4. Tableau récapitulatif des paramètres du modèle

| Symbole | Type | Description |
|---|---|---|
| $V$ | Ensemble | Nœuds du graphe (villes + dépôt $v_0$) |
| $v_0$ | Sommet | Nœud du graphe qui représente le dépôt |
| $E$ | Ensemble | Arêtes du graphe (routes disponibles) |
| $c_{ij}$ | Paramètre $\in \mathbb{R}^+$ | Coût (distance) de l'arête $(i,j)$ |
| $F \subseteq E$ | Ensemble | Arêtes interdites |
| $D \subseteq V^2$ | Ensemble | Paires de dépendance (livraison avant collecte) |
| $x_{ij}$ | Variable binaire $\in \{0,1\}$ | 1 si l'arête $(i,j)$ est utilisée dans la tournée |
| $p_i$ | Variable entière $\in \{1,\ldots,n\}$ | Rang de la ville $i$ dans l'ordre de visite |

# 5. Bibliographie

## 5.1 Ressources

1. [**LaTeX Documentation. (s. d.).**](https://www.latex-project.org/help/documentation/) - visité le 06/04/2026
2. [**Basic Syntax | Markdown Guide. (s. d.).**](https://www.markdownguide.org/basic-syntax/) - visité le 06/04/2026
2. [**Problème de vendeur de voyages. (s. d.).** *Google For Developers.*](https://developers.google.com/optimization/routing/tsp?hl=fr) - visité le 09/04/2026
3. [**Problème d’itinéraire. (s. d.).** *Google For Developers.*](https://developers.google.com/optimization/routing/vrp?hl=fr) - visité le 09/04/2026

## 5.2 Références Scientifiques

L'élaboration de cette modélisation s'appuie sur la littérature scientifique de la Recherche Opérationnelle :

5. [**Dantzig, G. B., & Ramser, J. H. (1959).** *The Truck Dispatching Problem*. Management Science, 6(1), 80-91.](https://pubsonline.informs.org/doi/10.1287/mnsc.6.1.80)
6. [**Miller, C. E., Tucker, A. W., & Zemlin, R. A. (1960).** *Integer Programming Formulation of Traveling Salesman Problems*. Journal of the ACM (JACM), 7(4), 326-329.](https://dl.acm.org/doi/10.1145/321043.321046)
7. [**Diego. (2020, 24 juin).** *Traveling Salesman Problem (TSP) with Miller-Tucker-Zemlin (MTZ) in CPLEX/OPL – CoEnzyme.*](https://co-enzyme.fr/blog/traveling-salesman-problem-tsp-in-cplex-opl-with-miller-tucker-zemlin-mtz-formulation/)
8. [**Teaching Integer Programming Formulations Using the Traveling Salesman Problem. (2003).** *SIAM REVIEW, 4.*](https://mate.unipv.it/~gualandi/famo2conti/papers/teaching_tsp.pdf)
9. [**Garey, M. R., & Johnson, D. S. (1979).** *Computers and Intractability: A Guide to the Theory of NP-Completeness. W. H. Freeman & Co.*](https://perso.limos.fr/~palafour/PAPERS/PDF/Garey-Johnson79.pdf)
10. [**Karp, R. M. (1972).** *Reducibility among combinatorial problems*. In Complexity of computer computations (pp. 85-103). Springer, Boston, MA.](https://www.cs.umd.edu/~gasarch/BLOGPAPERS/Karp.pdf)